# TAG-SEQ ANALYSIS - Clonal Fusion Experiment

- Use environment_transcriptomic.yml


- 2 cell lines (1806, 231)
- Parent clonal populations + matched homotypic fusion progeny
- **Option B (Matched)**: each fusion clone vs its two matched parent clones

---
### Updates
- **Option B (Matched)** fits one `~ clone` model per trio and extracts F-vs-P1, F-vs-P2, P1-vs-P2, and F-vs-mid-parent-value (MPV). See the MPV / expression-dominance classification section for additive / dominant / transgressive gene calls.
- New cells are marked `[NEW]` and revised cells `[REVISED]` in their header comments throughout.


In [ ]:
# ==============================================================================
#  FIGURE STYLE SETTINGS
# ==============================================================================
import os
import pathlib
from matplotlib import font_manager
from matplotlib import rcParams

# FONT_DIR = pathlib.Path('/stor/work/Brock/kennedy/fonts/arial')
# if FONT_DIR.exists():
#     for _fp in list(FONT_DIR.glob('*.TTF')) + list(FONT_DIR.glob('*.ttf')):
#         font_manager.fontManager.addfont(str(_fp))
#     _avail = [f.name for f in font_manager.fontManager.ttflist]
#     if 'Arial' in _avail:
#         print(f"Arial loaded from {FONT_DIR}")
#     else:
#         print(f"Font files found in {FONT_DIR} but Arial was not registered - using default font.")
# else:
#     print(f"Font dir not found ({FONT_DIR}) - using default font.")

# FONT_FAMILY = "Arial"
FONT_SIZE   = 16
FONT_BOLD   = False
FIG_EXT     = ".png"   # ".svg" or ".png"
FIG_DPI     = 300       # only used when FIG_EXT == ".png"

def apply_style():
    w = "bold" if FONT_BOLD else "normal"
    rcParams.update({
        # "font.family":           FONT_FAMILY,
        "font.size":             FONT_SIZE,
        "font.weight":           w,
        "axes.titlesize":        FONT_SIZE + 1,
        "axes.titleweight":      w,
        "axes.labelsize":        FONT_SIZE,
        "axes.labelweight":      w,
        "xtick.labelsize":       FONT_SIZE - 1,
        "ytick.labelsize":       FONT_SIZE - 1,
        "legend.fontsize":       FONT_SIZE - 1,
        "legend.title_fontsize": FONT_SIZE,
        "figure.titlesize":      FONT_SIZE + 2,
        "figure.titleweight":    w,
    })

apply_style()

def save_fig(fig, name, tight=True):
    path = os.path.join(OUT_DIR, f"{name}{FIG_EXT}")
    kw = {"bbox_inches": "tight"} if tight else {}
    if FIG_EXT.lower() == ".png":
        kw["dpi"] = FIG_DPI
    fig.savefig(path, **kw)
    print(f"Saved: {path}")
    return path

print(f"Style applied  |  FIG_EXT={FIG_EXT}  |  FONT_SIZE={FONT_SIZE}  |  FONT_BOLD={FONT_BOLD}")


In [ ]:
# %% ============================================================
# CELL 1 - IMPORTS & SETUP
# ============================================================

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from adjustText import adjust_text
from matplotlib_venn import venn2, venn3
import mygene

# rpy2 bridge
from rpy2.robjects import pandas2ri, r, globalenv
import rpy2.robjects as ro
%load_ext rpy2.ipython
pandas2ri.activate()

# -- paths ------------------------------------------------------------------
# UPDATE this path to point at your new experiment's salmon output
COUNTS_FILE = (
    "/stor/work/Brock/kennedy/SC_repo/data/TranscriptomicData/salmon.merged.gene_counts.tsv"
)
OUT_DIR    = "analysis_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# -- experiment design ------------------------------------------------------
CELL_LINES = ["1806", "231"]
CL_DISPLAY = {"1806": "HCC1806", "231": "MDA-MB-231"}

# PAIR_MAP defines the matched parent -> fusion relationships.
# Each entry: (fusion_clone_name, [parent1_name, parent2_name])
# Clone names must match exactly what appears in sample column names.
PAIR_MAP = {
    "231": [
        ("F_C1C4", ["C1", "C4"]),
        ("F_C6C8", ["C6", "C8"]),
    ],
    "1806": [
        ("F_C2C4", ["C2", "C4"]),
        ("F_C6C7", ["C6", "C7"]),
        ("F_C5C2", ["C5", "C2"]),
    ],
}

# Colour palettes
STATUS_PALETTE = {"Parent": "#000000", "Fusion": "#8B4513"}
# Per-pair colours for Venn / matched contrast plots
PAIR_COLOURS = [
    "#E07B54", "#5B8DB8", "#6DBF8A", "#C97AC9", "#D4A84B"
]

print("Setup complete. Output directory:", OUT_DIR)


In [ ]:
# %% ============================================================
# CELL 2 - LOAD COUNTS MATRIX
# ============================================================
# Expected input: salmon.merged.gene_counts.tsv
#   rows = genes, columns = samples (two gene-ID columns at left from nf-core)

counts_raw = pd.read_csv(COUNTS_FILE, sep="\t", index_col=1)
counts_raw = counts_raw.drop(columns=["gene_id"], errors="ignore")
counts_raw = np.round(counts_raw).astype(int)

print(f"Loaded counts: {counts_raw.shape[0]:,} genes x {counts_raw.shape[1]} samples")

# Deduplicate gene names - keep the row with highest total counts
# (handles PAR genes like CD99, VAMP7, SLC25A6 that appear on both X and Y)
counts_raw["_total"] = counts_raw.sum(axis=1)
counts_raw = counts_raw.sort_values("_total", ascending=False)
counts_raw = counts_raw[~counts_raw.index.duplicated(keep="first")]
counts_raw = counts_raw.drop(columns="_total")

print(f"After deduplication: {counts_raw.shape[0]:,} genes x {counts_raw.shape[1]} samples")
print(f"All gene names unique: {counts_raw.index.is_unique}")
counts_raw.head(3)


In [ ]:
# %% ============================================================
# CELL 3 - SAMPLE METADATA & CONDITION PARSING
# ============================================================
# Expected naming convention:
#   <CloneName>_<CellLine>_<Replicate>
#   e.g.  C1_231_a   F_C1C4_231_b   C2_1806_c   F_C5C2_1806_a
#
# Clone names:
#   Parent clones  -> Cx  (e.g. C1, C4, C6)
#   Fusion clones  -> F_CxCy  (e.g. F_C1C4, F_C6C8)

def parse_sample(name: str) -> dict:
    """Return {'cell_line', 'clone', 'fusion_status'} for a sample column name."""
    # -- cell line ----------------------------------------------------------
    if "1806" in name:
        cell_line = "1806"
    elif "231" in name:
        cell_line = "231"
    else:
        cell_line = "Unknown"

    # -- clone name: everything before the cell-line token -----------------
    parts  = name.split("_")
    cl_idx = next((i for i, p in enumerate(parts) if cell_line in p), None)
    if cl_idx is not None and cl_idx > 0:
        clone = "_".join(parts[:cl_idx])
    else:
        clone = "Unknown"

    # -- fusion status ------------------------------------------------------
    if clone.upper().startswith("F_"):
        fusion_status = "Fusion"
    elif clone.upper().startswith("C"):
        fusion_status = "Parent"
    else:
        fusion_status = "Unknown"

    return {"cell_line": cell_line, "clone": clone, "fusion_status": fusion_status}


coldata = pd.DataFrame(
    [parse_sample(c) for c in counts_raw.columns],
    index=counts_raw.columns
)

bad = (coldata["cell_line"] == "Unknown") | (coldata["clone"] == "Unknown")
if bad.any():
    print("WARNING - could not parse these samples, dropping them:")
    print(counts_raw.columns[bad].tolist())

counts_df = counts_raw.loc[:, ~bad]
coldata   = coldata[~bad]

print("Samples parsed:")
print(coldata.groupby(["cell_line", "fusion_status", "clone"]).size()
      .rename("n_replicates").to_string())

# Split into per-cell-line dicts for convenience
counts_by_cl = {
    cl: counts_df.loc[:, coldata["cell_line"] == cl]
    for cl in CELL_LINES
}
coldata_by_cl = {
    cl: coldata[coldata["cell_line"] == cl]
    for cl in CELL_LINES
}


In [ ]:
# %% ============================================================
# CELL 4 - MAP ENTREZ IDS TO GENE SYMBOLS (if needed)
# ============================================================

mg = mygene.MyGeneInfo()

numeric_genes = [g for g in counts_df.index if str(g)[0].isdigit()]
print(f"Found {len(numeric_genes)} numeric gene IDs to map")

if numeric_genes:
    results_mg = mg.querymany(
        numeric_genes,
        scopes  = "entrezgene",
        fields  = "symbol",
        species = "human",
        verbose = False
    )
    entrez_to_symbol = {
        str(hit["query"]): hit["symbol"]
        for hit in results_mg
        if "symbol" in hit and "notfound" not in hit
    }
    print(f"Mapped {len(entrez_to_symbol)} IDs")

    counts_df.index = [entrez_to_symbol.get(str(g), str(g)) for g in counts_df.index]
    counts_df["_total"] = counts_df.sum(axis=1)
    counts_df = counts_df.sort_values("_total", ascending=False)
    counts_df = counts_df[~counts_df.index.duplicated(keep="first")]
    counts_df = counts_df.drop(columns="_total")
    print(f"Final counts matrix: {counts_df.shape[0]:,} genes")

# Rebuild per-cell-line splits with updated gene names
counts_by_cl = {
    cl: counts_df.loc[:, coldata["cell_line"] == cl]
    for cl in CELL_LINES
}
print("Gene name mapping complete.")


In [ ]:
# %% ============================================================
# CELL 5 - QC: LIBRARY SIZE BAR CHART
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 5), constrained_layout=True)

for ax, cl in zip(axes, CELL_LINES):
    sub_cd = coldata_by_cl[cl]
    sub_ct = counts_by_cl[cl]
    lib_sizes = sub_ct.sum(axis=0) / 1e6

    colors = [STATUS_PALETTE.get(s, "grey") for s in sub_cd["fusion_status"]]
    ax.bar(range(len(lib_sizes)), lib_sizes, color=colors,
           edgecolor="white", linewidth=0.5)
    ax.set_xticks(range(len(lib_sizes)))
    ax.set_xticklabels(sub_cd.index, rotation=90)
    ax.set_ylabel("Library size (M reads)")
    ax.set_title(f"{CL_DISPLAY[cl]} - library sizes")
    ax.axhline(lib_sizes.median(), color="black", ls="--", lw=1,
               label=f"Median {lib_sizes.median():.1f} M")

    handles = [mpatches.Patch(color=c, label=s)
               for s, c in STATUS_PALETTE.items()]
    ax.legend(handles=handles, loc="upper right")

save_fig(fig, "qc_library_sizes")
plt.show()


In [ ]:
# %% ============================================================
# CELL 6 - QC: DETECTED GENES PER SAMPLE
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

for ax, cl in zip(axes, CELL_LINES):
    sub_cd = coldata_by_cl[cl].copy()
    sub_ct = counts_by_cl[cl]
    sub_cd["detected"] = (sub_ct > 0).sum(axis=0).values

    sns.boxplot(
        data=sub_cd, x="fusion_status", y="detected",
        order=["Parent", "Fusion"],
        palette=STATUS_PALETTE, ax=ax, width=0.5, linewidth=1,
        flierprops=dict(marker="o", markersize=4)
    )
    sns.stripplot(
        data=sub_cd, x="fusion_status", y="detected",
        order=["Parent", "Fusion"],
        hue="clone", ax=ax, size=5, alpha=0.85, jitter=True, legend=False
    )
    ax.set_title(f"{CL_DISPLAY[cl]} - detected genes per sample")
    ax.set_xlabel("")
    ax.set_ylabel("Genes with count > 0")

save_fig(fig, "qc_detected_genes")
plt.show()


In [ ]:
# %% ============================================================
# CELL 7 - QC: PCA ON VST COUNTS
# ============================================================

SHOW_PCA_LABELS = True   # set False to hide sample labels

r("""
suppressPackageStartupMessages(library(DESeq2))

vst_pca <- function(counts, conditions) {
    conditions <- factor(conditions)
    if ("Parent" %in% levels(conditions))
        conditions <- relevel(conditions, ref = "Parent")
    colData <- data.frame(condition = conditions,
                          row.names = colnames(counts))
    dds <- DESeqDataSetFromMatrix(countData = counts,
                                  colData   = colData,
                                  design    = ~ condition)
    dds <- dds[rowSums(counts(dds) >= 5) >= 3, ]
    vst_mat <- assay(vst(dds, blind = TRUE))
    pca_res <- prcomp(t(vst_mat), scale. = FALSE)
    scores  <- as.data.frame(pca_res$x[, 1:4])
    varexp  <- as.numeric(summary(pca_res)$importance[2, 1:4] * 100)
    list(scores = scores, varexp = varexp)
}
""")

status_markers = {"Parent": "o", "Fusion": "^"}
fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)

for ax, cl in zip(axes, CELL_LINES):
    sub_cd = coldata_by_cl[cl]
    sub_ct = counts_by_cl[cl]

    # Build a dynamic clone colour map for this cell line
    clones      = sub_cd["clone"].unique()
    clone_cmap  = plt.get_cmap("tab10")
    clone_color = {c: clone_cmap(i / max(len(clones)-1, 1))
                   for i, c in enumerate(clones)}

    globalenv["counts_r"] = sub_ct
    globalenv["cond_r"]   = ro.StrVector(sub_cd["fusion_status"].tolist())

    pca_result = r("vst_pca(counts_r, cond_r)")
    scores_r   = pca_result.rx2("scores")
    scores     = pd.DataFrame(
        pandas2ri.rpy2py(scores_r),
        columns=list(scores_r.colnames),
        index=sub_cd.index
    )
    varexp = list(pca_result.rx2("varexp"))

    for sample, row in scores.iterrows():
        status = sub_cd.loc[sample, "fusion_status"]
        clone  = sub_cd.loc[sample, "clone"]
        ax.scatter(
            row["PC1"], row["PC2"],
            c=np.array([clone_color[clone]]),
            marker=status_markers.get(status, "o"),
            s=90, edgecolors="black", linewidth=0.5, zorder=3
        )

    x_vals, y_vals = scores["PC1"], scores["PC2"]
    x_pad = (x_vals.max() - x_vals.min()) * 0.20
    y_pad = (y_vals.max() - y_vals.min()) * 0.20
    ax.set_xlim(x_vals.min() - x_pad, x_vals.max() + x_pad)
    ax.set_ylim(y_vals.min() - y_pad, y_vals.max() + y_pad)

    if SHOW_PCA_LABELS:
        texts = []
        for sample, row in scores.iterrows():
            texts.append(ax.text(row["PC1"], row["PC2"], sample,
                                 fontsize=FONT_SIZE-8, color="black", zorder=4))
        try:
            adjust_text(texts, ax=ax,
                        arrowprops=dict(arrowstyle="-", lw=0.4, color="grey"))
        except Exception:
            pass

    ax.set_xlabel(f"PC1 ({varexp[0]:.1f}% variance)")
    ax.set_ylabel(f"PC2 ({varexp[1]:.1f}% variance)")
    ax.set_title(f"{CL_DISPLAY[cl]} - PCA (VST, blind)")
    ax.axhline(0, color="lightgrey", lw=0.5, zorder=0)
    ax.axvline(0, color="lightgrey", lw=0.5, zorder=0)

    # Legend: clone colours + status markers
    clone_handles = [mpatches.Patch(color=clone_color[c], label=c)
                     for c in clones]
    status_handles = [
        plt.Line2D([0], [0], marker=status_markers[s], color="w",
                   markerfacecolor="grey", markersize=8, label=s)
        for s in ["Parent", "Fusion"]
    ]
    ax.legend(handles=clone_handles + status_handles,
              frameon=False, loc="upper center", fontsize=FONT_SIZE-4,
              bbox_to_anchor=(0.5, -0.12), ncol=4)

save_fig(fig, "qc_pca")
plt.show()

In [ ]:
# %% ============================================================
# CELL 8 - QC: SAMPLE DISTANCE HEATMAP
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(18, 7), constrained_layout=True)

for ax, cl in zip(axes, CELL_LINES):
    sub_cd = coldata_by_cl[cl]
    sub_ct = counts_by_cl[cl]

    globalenv["counts_r"] = sub_ct
    globalenv["cond_r"]   = ro.StrVector(sub_cd["fusion_status"].tolist())

    vst_mat = r("""
        suppressPackageStartupMessages(library(DESeq2))
        conditions <- factor(cond_r)
        if ("Parent" %in% levels(conditions))
            conditions <- relevel(conditions, ref = "Parent")
        colData <- data.frame(condition = conditions,
                              row.names = colnames(counts_r))
        dds <- DESeqDataSetFromMatrix(countData = counts_r,
                                      colData   = colData,
                                      design    = ~ condition)
        dds <- dds[rowSums(counts(dds) >= 5) >= 3, ]
        as.data.frame(assay(vst(dds, blind = TRUE)))
    """)
    vst_mat.columns = sub_cd.index

    dist_mat = pd.DataFrame(
        np.sqrt(
            ((vst_mat.T.values[:, None, :] - vst_mat.T.values[None, :, :]) ** 2
             ).sum(axis=2)
        ),
        index=sub_cd.index, columns=sub_cd.index
    )

    # Two annotation bars: fusion_status and clone
    status_colors = sub_cd["fusion_status"].map(STATUS_PALETTE).rename("Status")
    clones        = sub_cd["clone"].unique()
    clone_cmap    = plt.get_cmap("tab10")
    clone_color   = {c: clone_cmap(i / max(len(clones)-1, 1))
                     for i, c in enumerate(clones)}
    clone_colors  = sub_cd["clone"].map(
        lambda c: "#{:02x}{:02x}{:02x}".format(
            *[int(x*255) for x in clone_color[c][:3]])
    ).rename("Clone")

    col_colors = pd.concat([status_colors, clone_colors], axis=1)

    sns.heatmap(
        dist_mat, ax=ax,
        cmap="Blues_r",
        xticklabels=True, yticklabels=True,
        linewidths=0.3, linecolor="white",
        cbar_kws={"shrink": 0.6, "label": "Euclidean dist (VST)"}
    )
    ax.set_title(f"{CL_DISPLAY[cl]} - sample distances")
    ax.tick_params(axis="both", labelsize=6)

save_fig(fig, "qc_sample_distances")
plt.show()


In [ ]:
# %% ============================================================
# CELL 9 - [REVISED] R: TRIO DESeq2 HELPER (~ clone model, for MPV/classification)
# ============================================================
# Two changes from the original version:
#
# 1) FILTERING: previously required only 3 of 9 samples (pooled across all
#    three groups) to have count >= 5. That let a gene through even if one
#    entire group (e.g. the fusion clone) was all zeros - which is exactly
#    what happened with UCHL1 in F_C1C4. Now filtering requires each of
#    P1, P2, and F to independently have at least `min_samples` replicates
#    with count >= `min_count`, so a gene that's essentially absent in any
#    one group gets dropped before fitting, regardless of which contrast
#    later uses it.
#
# 2) SHRINKAGE: trio_contrast() now applies lfcShrink(..., type = "ashr")
#    to every contrast (F_vs_P1, P2_vs_P1, F_vs_P2, F_vs_MPV) uniformly.
#    ashr operates on the results() output directly (via res=), so it
#    works the same way for named coefficients and numeric contrasts,
#    unlike apeglm, which only accepts a single named coef. Using ashr
#    for all four keeps the shrinkage method consistent across contrasts
#    rather than mixing apeglm and ashr magnitudes.
#
# Significance calls (padj) are UNCHANGED by this - shrinkage only affects
# the reported LFC point estimate, not the underlying Wald test.

r("""
suppressPackageStartupMessages(library(DESeq2))

run_trio_deseq <- function(counts, clone_labels, min_count = 5, min_samples = 2) {
    # clone_labels must use the generic values "P1", "P2", "F"
    clone <- factor(clone_labels, levels = c("P1", "P2", "F"))
    colData <- data.frame(clone = clone, row.names = colnames(counts))

    # --- per-group filter: require min_samples replicates >= min_count
    #     independently within EACH of P1, P2, and F ---
    keep_per_group <- sapply(levels(clone), function(g) {
        idx <- clone == g
        rowSums(counts[, idx, drop = FALSE] >= min_count) >= min_samples
    })
    keep <- apply(keep_per_group, 1, all)

    dds <- DESeqDataSetFromMatrix(countData = counts, colData = colData,
                                  design = ~ clone)
    dds <- dds[keep, ]
    DESeq(dds, quiet = TRUE)
}

trio_contrast <- function(dds, coef_name = NULL, numeric_contrast = NULL, shrink = TRUE) {
    if (!is.null(coef_name)) {
        res <- results(dds, name = coef_name, independentFiltering = TRUE)
    } else {
        res <- results(dds, contrast = numeric_contrast, independentFiltering = TRUE)
    }
    if (shrink) {
        res <- lfcShrink(dds, res = res, type = "ashr", quiet = TRUE)
    }
    as.data.frame(res)
}
""")
print("Trio DESeq2 helper loaded (per-group filtering + ashr shrinkage). Contrasts available:")
print("  F vs P1   -> coef 'clone_F_vs_P1'")
print("  P2 vs P1  -> coef 'clone_P2_vs_P1'")
print("  F vs P2   -> numeric contrast c(0, -1, 1)")
print("  F vs MPV  -> numeric contrast c(0, -0.5, 1)   [MPV = mean(log2 P1, log2 P2)]")

## MPV / expression-dominance classification (replaces old "matched" pooled-parent contrast)

The previous **Option B (Matched)** cell tested each fusion clone against its two parents *merged into one "Parent" group* - a single Fusion-vs-pooled-parent test that couldn't reveal which parent fusion resembles, couldn't flag transgressive genes, and lost power specifically for genes where the two parents differ a lot (inter-parent variance was inflating the pooled group's dispersion estimate). **It has been removed and replaced with the three cells below.**

Each trio (F, P1, P2) now gets ONE correctly-specified `~ clone` model, from which four contrasts are pulled: **F vs P1**, **F vs P2**, **P1 vs P2**, and **F vs mid-parent value (MPV)**. These combine into a per-gene call - additive / dominant toward one parent / transgressive (beyond both parents) / conserved / fusion-altered.

`results_matched` is still populated below (now from the F-vs-MPV contrast) so every downstream cell that already consumes it - Venn diagrams, the DE heatmap, exports - keeps working unchanged, just on the corrected statistics.

In [ ]:
# %% ============================================================
# CELL 10 - [REVISED] MPV / EXPRESSION-DOMINANCE: FIT TRIO MODELS
# ============================================================
# Only change from the original: _pull() now requests shrunk LFCs by
# default (shrink=TRUE is the default in trio_contrast, so no call-site
# change is strictly needed, but it's passed explicitly here for clarity
# and so it's easy to flip off for a quick unshrunk comparison run).

results_pairwise = {}
results_matched  = {}

for cl in CELL_LINES:
    results_pairwise[cl] = {}
    results_matched[cl]  = {}
    sub_cd = coldata_by_cl[cl]
    sub_ct = counts_by_cl[cl]

    for fusion_clone, parents in PAIR_MAP[cl]:
        p1, p2 = parents
        keep_clones = [p1, p2, fusion_clone]
        mask   = sub_cd["clone"].isin(keep_clones)
        ct_sub = sub_ct.loc[:, mask]
        cd_sub = sub_cd[mask]

        # Map real clone names -> generic P1/P2/F labels for the R model
        generic = cd_sub["clone"].map({p1: "P1", p2: "P2", fusion_clone: "F"})

        print(f"  Fitting {CL_DISPLAY[cl]} trio: {fusion_clone} "
              f"({p1} + {p2}) ...", end=" ")
        try:
            globalenv["counts_r"]     = ct_sub
            globalenv["clone_r"]      = ro.StrVector(generic.tolist())
            globalenv["gene_names_r"] = ro.StrVector(ct_sub.index.tolist())
            r("rownames(counts_r) <- gene_names_r")
            r("dds_trio <- run_trio_deseq(counts_r, clone_r)")

            def _pull(coef=None, contrast=None, shrink=True):
                if coef is not None:
                    r(f'res_tmp <- trio_contrast(dds_trio, coef_name = "{coef}", shrink = {"TRUE" if shrink else "FALSE"})')
                else:
                    globalenv["contrast_r"] = ro.FloatVector(contrast)
                    r(f'res_tmp <- trio_contrast(dds_trio, numeric_contrast = contrast_r, shrink = {"TRUE" if shrink else "FALSE"})')
                gene_names = list(r("rownames(res_tmp)"))
                out = r("res_tmp")
                out.index = [str(g) for g in gene_names]
                out = out[~out.index.duplicated(keep="first")]
                return out[~out["padj"].isna()].copy()

            trio_res = {
                "F_vs_P1":  _pull(coef="clone_F_vs_P1"),
                "P2_vs_P1": _pull(coef="clone_P2_vs_P1"),
                "F_vs_P2":  _pull(contrast=[0, -1, 1]),
                "F_vs_MPV": _pull(contrast=[0, -0.5, 1]),
            }
            results_pairwise[cl][fusion_clone] = trio_res
            results_matched[cl][f"{fusion_clone}_vs_MPV"] = trio_res["F_vs_MPV"]
            print(f"\u2713  {trio_res['F_vs_MPV'].shape[0]:,} genes")
        except Exception as e:
            print(f"\u2717  ERROR: {e}")

print("\nTrio models complete.")

In [ ]:
# %% ============================================================
# CELL 11 - [NEW] MPV / EXPRESSION-DOMINANCE: CLASSIFY EVERY GENE
# ============================================================
# Categories:
#   conserved               parents don't differ; fusion == parents
#   fusion-altered           parents don't differ; fusion != parents (no "dominance" concept applies)
#   additive                 parents differ; fusion == mid-parent value
#   dominant_P1 / dominant_P2   parents differ; fusion resembles one parent specifically
#   transgressive_up/down    parents differ; fusion is outside the range of BOTH parents
#   ambiguous_nonadditive     parents differ, fusion differs from MPV and from both parents,
#                             but still falls within the parental range

PADJ_THRESH_MPV = 0.05

def classify_trio(trio_res, padj_thresh=PADJ_THRESH_MPV):
    f_p1  = trio_res["F_vs_P1"][["log2FoldChange", "padj"]].add_prefix("f_p1_")
    f_p2  = trio_res["F_vs_P2"][["log2FoldChange", "padj"]].add_prefix("f_p2_")
    p2_p1 = trio_res["P2_vs_P1"][["log2FoldChange", "padj"]].add_prefix("p2_p1_")
    f_mpv = trio_res["F_vs_MPV"][["log2FoldChange", "padj"]].add_prefix("f_mpv_")

    merged = f_p1.join(f_p2, how="inner").join(p2_p1, how="inner").join(f_mpv, how="inner")

    def _classify(row):
        p_sig   = row["p2_p1_padj"] < padj_thresh
        mpv_sig = row["f_mpv_padj"] < padj_thresh
        p1_sig  = row["f_p1_padj"]  < padj_thresh
        p2_sig  = row["f_p2_padj"]  < padj_thresh

        if not p_sig:
            return "fusion-altered" if mpv_sig else "conserved"
        if not mpv_sig:
            return "additive"

        lo, hi = sorted([0.0, row["p2_p1_log2FoldChange"]])
        f_pos = row["f_p1_log2FoldChange"]

        if p1_sig and not p2_sig:
            return "dominant_P2"
        if p2_sig and not p1_sig:
            return "dominant_P1"
        if p1_sig and p2_sig:
            if f_pos > hi:
                return "transgressive_up"
            if f_pos < lo:
                return "transgressive_down"
            return "ambiguous_nonadditive"
        return "ambiguous_nonadditive"

    merged["category"] = merged.apply(_classify, axis=1)
    return merged


classification = {}   # classification[cl][fusion_clone] = merged DataFrame w/ category

tsv_dir_mpv = os.path.join(OUT_DIR, "de_tables")
os.makedirs(tsv_dir_mpv, exist_ok=True)

for cl in CELL_LINES:
    classification[cl] = {}
    for fusion_clone, trio_res in results_pairwise[cl].items():
        classification[cl][fusion_clone] = classify_trio(trio_res)
        out_path = os.path.join(
            tsv_dir_mpv, f"{CL_DISPLAY[cl]}_{fusion_clone}_MPV_classification.tsv"
        )
        classification[cl][fusion_clone].to_csv(out_path, sep="\t")

print("Classification complete.")


In [ ]:
# %% ============================================================
# CELL 12 - [NEW] MPV / EXPRESSION-DOMINANCE: SUMMARY TABLE & PLOT
# ============================================================

CATEGORY_PALETTE = {
    "conserved":             "#D9D9D9",
    "additive":              "#B0B0B0",
    "dominant_P1":           "#777777",
    "dominant_P2":           "#363636",
    "ambiguous_nonadditive": "#D4A84B",
    "fusion-altered":        "#6B5331",
    "transgressive_up":      "#B03A2E",
    "transgressive_down":    "#2E4E9E",
}
CATEGORY_ORDER_MPV = list(CATEGORY_PALETTE.keys())

CATEGORY_LABELS = {
    "conserved":             "Conserved",
    "additive":               "Additive",
    "dominant_P1":            "Dominant (P1)",
    "dominant_P2":            "Dominant (P2)",
    "ambiguous_nonadditive":  "Ambiguous Non-additive",
    "fusion-altered":         "Fusion-altered",
    "transgressive_up":       "Transgressive (Up)",
    "transgressive_down":     "Transgressive (Down)",
}

summary_rows = []
for cl in CELL_LINES:
    for fusion_clone, df in classification[cl].items():
        counts_by_cat = df["category"].value_counts()
        total = len(df)
        for cat in CATEGORY_ORDER_MPV:
            n = int(counts_by_cat.get(cat, 0))
            summary_rows.append({
                "cell_line": CL_DISPLAY[cl], "fusion_clone": fusion_clone,
                "category": cat, "n_genes": n,
                "pct": round(100 * n / total, 1) if total else 0.0,
            })

mpv_summary = pd.DataFrame(summary_rows)
mpv_summary.to_csv(os.path.join(OUT_DIR, "mpv_classification_summary.csv"), index=False)
print(mpv_summary.pivot(index=["cell_line", "fusion_clone"],
                        columns="category", values="pct").to_string())

# -- Stacked bar plot: % of genes per category, one bar per fusion clone --
pivot_pct = mpv_summary.pivot(index=["cell_line", "fusion_clone"],
                              columns="category", values="pct")[CATEGORY_ORDER_MPV]

def _bar_label(cl_display: str, fusion_clone: str) -> str:
    """'HCC1806', 'F_C1C4' -> 'HCC1806\nFusion\nC1C4'"""
    pair = fusion_clone[2:] if fusion_clone.upper().startswith("F_") else fusion_clone
    return f"{cl_display}\nFusion\n{pair}"

x_labels = [_bar_label(cl, fc) for cl, fc in pivot_pct.index]

# -- Plot layout controls -----------------------------------------------
FIG_HEIGHT     = 6      # figure height, in inches
COL_WIDTH      = 1    # figure width allotted per bar/column, in inches
BAR_WIDTH      = 0.8    # bar width as a fraction of its column slot (0-1); smaller = more space between bars
LEGEND_NCOL    = 2       # number of legend columns
LEGEND_ROW_PAD = 1.8     # vertical space per legend row, in units of the legend font size - increase for more row spacing
LEGEND_GAP     = 0.15    # extra fixed gap between the x-axis and the legend, in inches

legend_fontsize  = FONT_SIZE - 2
n_legend_rows    = -(-len(CATEGORY_ORDER_MPV) // LEGEND_NCOL)   # ceil division
legend_height_in = n_legend_rows * (legend_fontsize / 72) * LEGEND_ROW_PAD + LEGEND_GAP
bbox_y           = -(legend_height_in / FIG_HEIGHT)

fig, ax = plt.subplots(figsize=(max(7, COL_WIDTH * len(pivot_pct)), FIG_HEIGHT),
                        constrained_layout=True)
bottom = np.zeros(len(pivot_pct))
for cat in CATEGORY_ORDER_MPV:
    vals = pivot_pct[cat].values
    ax.bar(x_labels, vals,
           bottom=bottom, color=CATEGORY_PALETTE[cat], label=CATEGORY_LABELS[cat],
           edgecolor="white", linewidth=0.5, width=BAR_WIDTH)
    bottom += vals

ax.set_ylabel("% of tested genes")
ax.set_title("MPV / expression-dominance classification per fusion clone")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, bbox_y),
          ncol=LEGEND_NCOL, frameon=False, fontsize=FONT_SIZE-4)
ax.tick_params(axis="x", rotation=0)
ax.set_xticklabels(x_labels, ha="center",fontsize=FONT_SIZE-4)

save_fig(fig, "mpv_classification_stacked_bar")
plt.show()

In [ ]:
# %% ============================================================
# CELL 13 - DE SUMMARY TABLE
# ============================================================

LFC_THRESH  = 0.585   # |log2FC| cut-off  (~1.5-fold)
PADJ_THRESH = 0.05

rows = []
for cl in CELL_LINES:
    # Matched
    for key, df in results_matched[cl].items():
        if df.empty:
            continue
        sig = df[df["padj"] < PADJ_THRESH]
        up  = sig[sig["log2FoldChange"] >  LFC_THRESH]
        dn  = sig[sig["log2FoldChange"] < -LFC_THRESH]
        rows.append({"Cell line": CL_DISPLAY[cl], "Analysis": "Matched",
                     "Contrast": key, "Total sig": len(sig),
                     "Up": len(up), "Down": len(dn)})

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))
summary_df.to_csv(os.path.join(OUT_DIR, "de_summary_table.csv"), index=False)
summary_df


In [ ]:
# %% ============================================================
# CELL 14 - MA PLOTS
# ============================================================

# Gather all contrasts, matched per cell line
all_contrasts = []   # list of (cl, analysis_label, key, df)
for cl in CELL_LINES:
    for key, df in results_matched[cl].items():
        all_contrasts.append((cl, "Matched", key, df))

n = len(all_contrasts)
ncols = max(len(results_matched[cl]) for cl in CELL_LINES)
# Simple flat layout
fig, axes = plt.subplots(1, n, figsize=(4 * n, 5), constrained_layout=True)
if n == 1:
    axes = [axes]

for ax, (cl, analysis, key, df) in zip(axes, all_contrasts):
    if df.empty:
        ax.set_visible(False)
        continue
    x   = np.log10(df["baseMean"].clip(lower=1))
    y   = df["log2FoldChange"]
    sig = df["padj"] < PADJ_THRESH

    ax.scatter(x[~sig], y[~sig], color="lightgrey", s=3, alpha=0.6, rasterized=True)
    ax.scatter(x[sig & (y >  LFC_THRESH)], y[sig & (y >  LFC_THRESH)],
               color="firebrick",      s=5, alpha=0.85, rasterized=True)
    ax.scatter(x[sig & (y < -LFC_THRESH)], y[sig & (y < -LFC_THRESH)],
               color="cornflowerblue", s=5, alpha=0.85, rasterized=True)
    ax.scatter(x[sig & (np.abs(y) <= LFC_THRESH)],
               y[sig & (np.abs(y) <= LFC_THRESH)],
               color="orange", s=3, alpha=0.6, rasterized=True)

    ax.axhline(0,           color="black", lw=0.8)
    ax.axhline( LFC_THRESH, color="black", ls="--", lw=0.6, alpha=0.6)
    ax.axhline(-LFC_THRESH, color="black", ls="--", lw=0.6, alpha=0.6)
    ax.set_xlabel(r"$\log_{10}$ mean expression", fontsize=8)
    ax.set_ylabel(r"$\log_2$ fold change",        fontsize=8)
    ax.set_title(f"{CL_DISPLAY[cl]}\n{analysis}\n{key}", fontsize=7)

save_fig(fig, "ma_plots")
plt.show()


In [ ]:
# %% ============================================================
# CELL 15 - VOLCANO PLOTS
# ============================================================

PLABEL_THRESH = 1e-5
LFC_LABEL_MIN = LFC_THRESH

def _suppress_label(gene: str) -> bool:
    u = gene.upper()
    if u.startswith(("TRN", "SNAR", "SNOR", "RNR", "RNA", "MIR", "LINC",
                      "MT-", "RPS", "RPL", "MRPS", "MRPL")):
        return True
    return False


def volcano(ax, df, title="",
            lfc_thresh=LFC_THRESH, padj_thresh=PADJ_THRESH,
            plabel_thresh=PLABEL_THRESH):
    if df.empty:
        ax.set_visible(False)
        return

    x    = df["log2FoldChange"]
    y    = -np.log10(df["padj"].clip(lower=1e-300))
    xmax = np.nanmax(np.abs(x.values)) * 1.1

    ns  = df["padj"] >= padj_thresh
    up  = (df["padj"] < padj_thresh) & (x >  lfc_thresh)
    dn  = (df["padj"] < padj_thresh) & (x < -lfc_thresh)
    mid = (df["padj"] < padj_thresh) & (np.abs(x) <= lfc_thresh)

    ax.scatter(x[ns],  y[ns],  c="lightgrey",     s=3, alpha=0.7, rasterized=True)
    ax.scatter(x[mid], y[mid], c="orange",         s=4, alpha=0.8, rasterized=True)
    ax.scatter(x[up],  y[up],  c="firebrick",      s=6, alpha=0.9, rasterized=True)
    ax.scatter(x[dn],  y[dn],  c="cornflowerblue", s=6, alpha=0.9, rasterized=True)

    ymax = y.max() if y.max() > 1 else 1
    ax.text( xmax * 0.97, ymax * 0.97, f"↑ {up.sum()}",
             ha="right", va="top", fontsize=7, color="firebrick")
    ax.text(-xmax * 0.97, ymax * 0.97, f"↓ {dn.sum()}",
             ha="left",  va="top", fontsize=7, color="cornflowerblue")

    label_mask = (df["padj"] < plabel_thresh) & (np.abs(x) > lfc_thresh)
    texts = []
    for gene in df.index[label_mask]:
        if not _suppress_label(gene):
            texts.append(ax.text(
                df.at[gene, "log2FoldChange"],
                -np.log10(df.at[gene, "padj"]),
                gene, fontsize=5, color="black"
            ))
    if texts:
        try:
            adjust_text(texts, ax=ax,
                        arrowprops=dict(arrowstyle="-", lw=0.4, color="grey"))
        except Exception:
            pass

    ax.axhline(-np.log10(padj_thresh), color="black", ls="--", lw=0.7, alpha=0.6)
    ax.axvline( lfc_thresh,            color="black", ls="--", lw=0.7, alpha=0.6)
    ax.axvline(-lfc_thresh,            color="black", ls="--", lw=0.7, alpha=0.6)
    ax.set_xlim(-xmax, xmax)
    ax.set_xlabel(r"$\log_2$ fold change",           fontsize=8)
    ax.set_ylabel(r"$-\log_{10}$ (adjusted p-value)", fontsize=8)
    ax.set_title(title, fontsize=8)

# -- Matched: one panel per pair x cell line -------------------------------
n_matched = max(len(results_matched[cl]) for cl in CELL_LINES)
fig, axes = plt.subplots(len(CELL_LINES), n_matched,
                          figsize=(3.5 * n_matched, 7 * len(CELL_LINES)),
                          constrained_layout=True)
axes = np.array(axes).reshape(len(CELL_LINES), n_matched)

for row_idx, cl in enumerate(CELL_LINES):
    matched_items = list(results_matched[cl].items())
    for col_idx in range(n_matched):
        ax = axes[row_idx, col_idx]
        if col_idx >= len(matched_items):
            ax.set_visible(False)
            continue
        key, df = matched_items[col_idx]
        volcano(ax, df, title=f"{CL_DISPLAY[cl]}\n{key}")

fig.suptitle("Option B - Matched volcano plots", fontsize=11)
save_fig(fig, "volcano_matched")
plt.show()


In [ ]:
# %% ============================================================
# CELL 16 - DE HEATMAP (top N significant genes)
# One heatmap per cell line; genes drawn from the matched results.
# ============================================================

TOP_N = 50   # top N genes per contrast (by padj)
ANNOTATION_ROW_HEIGHT_IN = 0.3   # thickness of the Clone/Status color bars, in inches - stays fixed regardless of gene-row height
DENDRO_LINEWIDTH = 1.5            # thickness of the dendrogram branch lines
SHOW_CLONE_ROW   = False           # set False to drop the Clone annotation row (and its legend entries)
LEGEND_GAP_ABOVE_CBAR = 0.02      # vertical gap between colorbar and legend, in figure-fraction units
LEGEND_NCOL = 1                   # number of columns in the Status/Clone legend

for cl in CELL_LINES:
    top_genes = set()

    # Collect top genes from each matched result
    for key, df in results_matched[cl].items():
        if df.empty:
            continue
        sig = df[(df["padj"] < PADJ_THRESH) & (np.abs(df["log2FoldChange"]) > LFC_THRESH)]
        top_genes.update(sig.nsmallest(TOP_N, "padj").index.tolist())

    if not top_genes:
        print(f"No significant genes for {CL_DISPLAY[cl]} - skipping heatmap.")
        continue

    top_genes = sorted(top_genes)
    sub_ct    = counts_by_cl[cl]
    sub_cd    = coldata_by_cl[cl]

    globalenv["counts_r"] = sub_ct
    globalenv["cond_r"]   = ro.StrVector(sub_cd["fusion_status"].tolist())

    vst_mat = r("""
        suppressPackageStartupMessages(library(DESeq2))
        conditions <- factor(cond_r)
        if ("Parent" %in% levels(conditions))
            conditions <- relevel(conditions, ref = "Parent")
        colData <- data.frame(condition = conditions,
                              row.names = colnames(counts_r))
        dds <- DESeqDataSetFromMatrix(countData = counts_r,
                                      colData   = colData,
                                      design    = ~ condition)
        dds <- dds[rowSums(counts(dds) >= 5) >= 3, ]
        as.data.frame(assay(vst(dds, blind = TRUE)))
    """)
    vst_mat.columns = sub_cd.index

    plot_genes = [g for g in top_genes if g in vst_mat.index]
    if not plot_genes:
        print(f"No top genes survived VST filtering for {CL_DISPLAY[cl]}")
        continue

    hm_z = vst_mat.loc[plot_genes].apply(
        lambda row: (row - row.mean()) / (row.std() + 1e-9), axis=1
    )

    # Two column annotation bars: fusion status + clone
    clones     = sub_cd["clone"].unique()
    clone_cmap = plt.get_cmap("tab10")
    clone_hex  = {c: "#{:02x}{:02x}{:02x}".format(
                      *[int(x*255) for x in clone_cmap(i / max(len(clones)-1, 1))[:3]])
                  for i, c in enumerate(clones)}

    col_colors_dict = {"Status": sub_cd["fusion_status"].map(STATUS_PALETTE)}
    if SHOW_CLONE_ROW:
        col_colors_dict["Clone"] = sub_cd["clone"].map(clone_hex)
    col_colors = pd.DataFrame(col_colors_dict)

    fig_width  = max(12, len(sub_cd) * 0.4)
    fig_height = max(10, len(plot_genes) * 0.25)
    annotation_ratio = ANNOTATION_ROW_HEIGHT_IN / fig_height

    g = sns.clustermap(
        hm_z,
        col_colors   = col_colors,
        cmap         = "RdBu_r",
        vmin=-2, vmax=2,
        yticklabels  = True,
        xticklabels  = True,
        figsize      = (fig_width, fig_height),
        dendrogram_ratio = (0.1, 0.05),
        colors_ratio = annotation_ratio,
        cbar_pos     = None,
        linewidths   = 0,
        rasterized   = True
    )

    for dendro_ax in [g.ax_row_dendrogram, g.ax_col_dendrogram]:
        for coll in dendro_ax.collections:
            coll.set_linewidth(DENDRO_LINEWIDTH)

    heatmap_pos = g.ax_heatmap.get_position()
    cbar_ax = g.fig.add_axes([
        heatmap_pos.x1 + 0.15,   # just right of the heatmap
        heatmap_pos.y0,          # aligned to heatmap bottom
        0.02,                    # width
        heatmap_pos.height*0.25       # matches heatmap height
    ])
    plt.colorbar(g.ax_heatmap.collections[0], cax=cbar_ax)
    cbar_ax.set_ylabel("Z-score")
    cbar_ax.tick_params(labelsize=FONT_SIZE - 4)

    # Status/Clone legend - anchored just above the colorbar, wherever it ends up
    legend_handles = [mpatches.Patch(color=color, label=status)
                       for status, color in STATUS_PALETTE.items()]
    if SHOW_CLONE_ROW:
        legend_handles += [mpatches.Patch(color=color, label=clone)
                            for clone, color in clone_hex.items()]

    cbar_bbox = cbar_ax.get_position()
    g.fig.legend(
        handles=legend_handles,
        loc="lower center",
        bbox_to_anchor=(cbar_bbox.x0 + cbar_bbox.width / 2 + 0.05,
                        cbar_bbox.y1 + LEGEND_GAP_ABOVE_CBAR),
        ncol=LEGEND_NCOL, frameon=False, fontsize=FONT_SIZE - 4
    )

    g.ax_heatmap.set_xticklabels(
        g.ax_heatmap.get_xticklabels(), rotation=90, fontsize=FONT_SIZE - 4)
    g.ax_heatmap.set_yticklabels(
        g.ax_heatmap.get_yticklabels(), rotation=0, fontsize=FONT_SIZE - 4)

    g.fig.suptitle(
        f"{CL_DISPLAY[cl]} - top DE genes (matched, |LFC|>{LFC_THRESH}, padj<{PADJ_THRESH})",
        y=1.01, fontsize=10
    )
    save_fig(g.fig, f"heatmap_{cl}")
    plt.show()

In [ ]:
# %% ============================================================
# CELL 17 - VENN DIAGRAMS: DE GENE OVERLAP ACROSS MATCHED PAIRS
# Within each cell line, shows which DE genes are shared across
# independent fusion events (2-way for 231, 3-way for 1806).
# ============================================================

def _venn_label(key: str) -> str:
    """'F_C1C4_vs_MPV' -> 'Fusion C1C4'"""
    fusion_clone = key.replace("_vs_MPV", "")
    pair = fusion_clone[2:] if fusion_clone.upper().startswith("F_") else fusion_clone
    return f"Fusion {pair}"

for cl in CELL_LINES:
    matched_keys = list(results_matched[cl].keys())
    n_pairs      = len(matched_keys)

    if n_pairs < 2:
        print(f"{CL_DISPLAY[cl]}: only {n_pairs} matched pair - Venn skipped.")
        continue

    # Build gene sets
    sets   = []
    labels = []
    for i, key in enumerate(matched_keys):
        df = results_matched[cl].get(key, pd.DataFrame())
        if df.empty:
            sets.append(set())
        else:
            sig = df[(df["padj"] < PADJ_THRESH) &
                     (np.abs(df["log2FoldChange"]) > LFC_THRESH)]
            sets.append(set(sig.index))
        labels.append(_venn_label(key))

    fig, ax = plt.subplots(1, 1, figsize=(6, 5), constrained_layout=True)
    colors  = PAIR_COLOURS[:n_pairs]

    if n_pairs == 2:
        venn2(sets, set_labels=labels, ax=ax,
              set_colors=colors, alpha=0.55)
    elif n_pairs == 3:
        venn3(sets, set_labels=labels, ax=ax,
              set_colors=colors, alpha=0.55)
    else:
        ax.text(0.5, 0.5, f"{n_pairs} pairs - Venn not supported for >3 sets",
                transform=ax.transAxes, ha="center")

    ax.set_title(
        f"{CL_DISPLAY[cl]} - DE gene overlap across matched fusion pairs\n"
        f"(|LFC| > {LFC_THRESH}, padj < {PADJ_THRESH})"
    )
    save_fig(fig, f"venn_matched_pairs_{cl}")
    plt.show()

In [ ]:
# %% ============================================================
# CELL 18 - PRINT OVERLAPPING GENES FROM MATCHED-PAIR VENNS
# ============================================================

from itertools import combinations

for cl in CELL_LINES:
    matched_keys = list(results_matched[cl].keys())
    n_pairs      = len(matched_keys)

    print(f"\n{'='*70}")
    print(f"  {CL_DISPLAY[cl]} - matched-pair DE gene overlaps")
    print(f"{'='*70}")

    # Build gene sets
    sets = {}
    for key in matched_keys:
        df = results_matched[cl].get(key, pd.DataFrame())
        if df.empty:
            sets[key] = set()
        else:
            sig = df[(df["padj"] < PADJ_THRESH) &
                     (np.abs(df["log2FoldChange"]) > LFC_THRESH)]
            sets[key] = set(sig.index)
        label = key.replace("_vs_parents", "")
        print(f"  {label}: {len(sets[key])} significant genes")

    # All pairwise overlaps
    for key_a, key_b in combinations(matched_keys, 2):
        label_a = key_a.replace("_vs_parents", "")
        label_b = key_b.replace("_vs_parents", "")
        shared = sets[key_a] & sets[key_b]
        only_a = sets[key_a] - sets[key_b]
        only_b = sets[key_b] - sets[key_a]

        print(f"\n  {label_a}  vs  {label_b}:")
        print(f"\n    Shared ({len(shared)} genes):")
        if shared:
            for gene in sorted(shared):
                lfc_a = results_matched[cl][key_a].at[gene, "log2FoldChange"]
                lfc_b = results_matched[cl][key_b].at[gene, "log2FoldChange"]
                print(f"      {gene:<20}  {label_a} LFC={lfc_a:+.2f}  "
                      f"{label_b} LFC={lfc_b:+.2f}")
        else:
            print("      None")
        print(f"\n    Only in {label_a} ({len(only_a)} genes):")
        if only_a:
            print(f"      {', '.join(sorted(only_a))}")
        print(f"\n    Only in {label_b} ({len(only_b)} genes):")
        if only_b:
            print(f"      {', '.join(sorted(only_b))}")

    # Triple overlap if 3 pairs
    if n_pairs >= 3:
        all_sets = list(sets.values())
        triple   = all_sets[0] & all_sets[1] & all_sets[2]
        print(f"\n  Shared across ALL three matched pairs ({len(triple)} genes):")
        if triple:
            for gene in sorted(triple):
                lfc_vals = "  ".join(
                    f"{k.replace('_vs_parents', '')} LFC="
                    f"{results_matched[cl][k].at[gene, 'log2FoldChange']:+.2f}"
                    for k in matched_keys if gene in results_matched[cl][k].index
                )
                print(f"    {gene:<20}  {lfc_vals}")
        else:
            print("    None")


In [ ]:
# %% ============================================================
# CELL 19 - [NEW] EXPORT ALL DE GENES PER FUSION CLONE (F vs MPV)
# All significant genes per fusion clone (raw F-vs-MPV, padj < PADJ_THRESH
# & |LFC| > LFC_THRESH) - not restricted to genes unique to that clone.
# Includes the MPV classification category, direction, and a
# "shared_with" column showing which other fusion clones (if any) also
# have this gene significant, so overlap status is visible in the table
# without needing to cross-reference the Venn separately.
# ============================================================

de_export_dir = os.path.join(OUT_DIR, "de_genes_fusion_vs_mpv")
os.makedirs(de_export_dir, exist_ok=True)

for cl in CELL_LINES:
    fusion_clones = list(classification[cl].keys())

    print(f"\n{'='*60}")
    print(f"  {CL_DISPLAY[cl]} - DE genes per fusion clone (F vs MPV)")
    print(f"{'='*60}")

    # Build significant gene sets per fusion clone (raw F vs MPV)
    sig_sets = {}
    for fusion_clone in fusion_clones:
        df = classification[cl][fusion_clone]
        sig = df[
            (df["f_mpv_padj"] < PADJ_THRESH) &
            (df["f_mpv_log2FoldChange"].abs() > LFC_THRESH)
        ]
        sig_sets[fusion_clone] = set(sig.index)

    for fusion_clone in fusion_clones:
        other_clones = [fc for fc in fusion_clones if fc != fusion_clone]

        df = classification[cl][fusion_clone]
        sig_genes = sig_sets[fusion_clone]
        if not sig_genes:
            print(f"  {fusion_clone}: no significant genes")
            continue

        df_full = df.loc[
            list(sig_genes),
            ["f_mpv_log2FoldChange", "f_mpv_padj", "category",
             "f_p1_log2FoldChange", "f_p2_log2FoldChange"]
        ].rename(columns={
            "f_mpv_log2FoldChange": "log2FoldChange",
            "f_mpv_padj":           "padj",
        }).copy()
        df_full = df_full.sort_values("log2FoldChange", ascending=False)

        df_full["direction"] = np.where(df_full["log2FoldChange"] > 0, "up", "down")
        df_full["shared_with"] = [
            ", ".join(fc for fc in other_clones if gene in sig_sets[fc]) or "none"
            for gene in df_full.index
        ]
        df_full["contrast"]  = f"{fusion_clone}_vs_MPV"
        df_full["cell_line"] = CL_DISPLAY[cl]

        n_up, n_down = (df_full["direction"] == "up").sum(), (df_full["direction"] == "down").sum()
        n_shared = (df_full["shared_with"] != "none").sum()
        print(f"\n  {fusion_clone}: {len(sig_genes)} DE genes "
              f"({n_up} up, {n_down} down; {n_shared} shared with \u22651 other clone)")
        print(f"    category breakdown: {df_full['category'].value_counts().to_dict()}")

        fname = os.path.join(
            de_export_dir, f"{CL_DISPLAY[cl]}_{fusion_clone}_DE_vs_MPV.tsv"
        )
        df_full.to_csv(fname, sep="\t")
        print(f"  Saved: {fname}")

print("\nDone.")

In [ ]:
# %% ============================================================
# CELL 20 - EXPORT: DE TABLES + MANIFEST
# ============================================================

import openpyxl

tsv_dir = os.path.join(OUT_DIR, "de_tables")
os.makedirs(tsv_dir, exist_ok=True)

# -- TSV per contrast --------------------------------------------
for cl in CELL_LINES:
    for key, df in results_matched[cl].items():
        if df.empty: continue
        df.to_csv(os.path.join(tsv_dir, f"{CL_DISPLAY[cl]}_matched_{key}.tsv"), sep="\t")

print(f"TSV tables written to {tsv_dir}/")

# -- Excel workbook ---------------------------------------------
xlsx_path = os.path.join(OUT_DIR, "all_de_results.xlsx")
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    for cl in CELL_LINES:
        for key, df in results_matched[cl].items():
            if df.empty: continue
            sheet = f"{cl}_{key}"[:31]
            df.sort_values("padj").to_excel(writer, sheet_name=sheet)
print(f"Excel workbook written: {xlsx_path}")

# -- Manifest ----------------------------------------------------
manifest_rows = []
for root, dirs, files in os.walk(OUT_DIR):
    for f in sorted(files):
        full = os.path.join(root, f)
        manifest_rows.append({"file": os.path.relpath(full, OUT_DIR),
                               "size_kb": os.path.getsize(full) // 1024})
manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(os.path.join(OUT_DIR, "output_manifest.csv"), index=False)
print("\nAll outputs:")
print(manifest.to_string(index=False))

In [ ]:
# %% ============================================================
# CELL 21 - COMPILE ALL CONTRASTS INTO ONE TSV PER CELL LINE
# Matched results combined; LFC filled 0, padj filled 1 for missing.
# ============================================================

for cl in CELL_LINES:
    compiled = None
    all_results_cl = {
        **{f"matched_{k}": v for k, v in results_matched[cl].items()},
    }

    for key, df in all_results_cl.items():
        if df.empty:
            continue
        df_sub = df[["log2FoldChange", "padj"]].rename(columns={
            "log2FoldChange": f"{key}_lfc",
            "padj":           f"{key}_p"
        })
        compiled = df_sub if compiled is None else compiled.join(df_sub, how="outer")

    if compiled is None:
        print(f"No results for {CL_DISPLAY[cl]}, skipping.")
        continue

    for col in compiled.columns:
        if col.endswith("_lfc"):
            compiled[col] = compiled[col].fillna(0)
        elif col.endswith("_p"):
            compiled[col] = compiled[col].fillna(1)

    out_path = os.path.join(OUT_DIR, "de_tables",
                            f"{CL_DISPLAY[cl]}_all_contrasts_compiled.tsv")
    compiled.to_csv(out_path, sep="\t")
    print(f"{CL_DISPLAY[cl]}: {len(all_results_cl)} contrasts, "
          f"{len(compiled):,} genes -> {out_path}")

print("\nDone.")
